## Cell 1: Imports & Configuration

In [27]:
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO
from deepface import DeepFace

# ============================================================
#  CONFIGURATION — Edit these paths before running
# ============================================================
MODEL_PATH      = r'D:\Major_project\model\yolov8l-face-lindevs.pt'
KNOWN_FACES_DIR = r'D:\Major_project\known_faces'
VIDEO_SOURCE    = 1        # 0 = webcam  |  or a file path string

CONF_THRESHOLD          = 0.5   # YOLO confidence
RECOGNITION_THRESHOLD   = 0.55  # ArcFace cosine-distance (lower = stricter)
NEW_FACE_CONFIRM_FRAMES = 5     # frames before a new face enters tracker
IOU_THRESHOLD           = 0.25  # IOU to link detections to tracked faces
# Re-recognition queue — one face processed per frame
# Red boxes have priority (jump to front of queue)
# All faces cycle through; no strict timer needed
RERECOG_QUEUE_COOLDOWN  = 1.0   # min seconds before same face re-enters queue

print('Configuration loaded.')


Configuration loaded.


## Cell 2: Utility / Helper Functions

In [28]:
def compute_iou(boxA, boxB):
    """Intersection over Union — both boxes in (x1,y1,x2,y2)."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / float(areaA + areaB - inter)


def xyxy_to_xywh(box):
    x1,y1,x2,y2 = box
    return (x1, y1, x2-x1, y2-y1)


def xywh_to_xyxy(box):
    x,y,w,h = box
    return (int(x), int(y), int(x+w), int(y+h))


def clamp_box(box, frame_shape):
    h,w = frame_shape[:2]
    x1,y1,x2,y2 = box
    return (max(0,x1), max(0,y1), min(w,x2), min(h,y2))


def is_unknown(label):
    """Return True if label represents an unrecognised face."""
    return label.startswith('Unknown')


print('Utility functions defined.')

Utility functions defined.


## Cell 3: Face Recognition Functions

In [29]:
def compute_arcface_embedding(face_crop):
    """
    Generate ArcFace embedding for a BGR numpy face crop.
    Returns 512-d numpy array or None on failure.
    """
    try:
        objs = DeepFace.represent(
            img_path=face_crop,
            model_name='ArcFace',
            enforce_detection=True,
            detector_backend='mtcnn'
        )
        return np.array(objs[0]['embedding'])
    except Exception:
        return None


def find_best_match(target_emb, known_encodings, known_names, threshold):
    """
    Compare target embedding against all known averaged embeddings.
    Each entry in known_encodings is already an averaged embedding.
    Returns (name, distance).
    """
    if len(known_encodings) == 0 or target_emb is None:
        return 'Unknown', 1.0

    t_norm     = np.linalg.norm(target_emb)
    best_name  = 'Unknown'
    min_dist   = float('inf')

    for emb, name in zip(known_encodings, known_names):
        k_norm   = np.linalg.norm(emb)
        sim      = np.dot(target_emb, emb) / (t_norm * k_norm + 1e-9)
        dist     = 1.0 - sim
        if dist < min_dist:
            min_dist = dist
            if dist <= threshold:
                best_name = name

    return best_name, min_dist


def recognize_face(face_crop, known_encodings, known_names, threshold):
    """Crop → embedding → match. Returns (name, distance)."""
    emb = compute_arcface_embedding(face_crop)
    return find_best_match(emb, known_encodings, known_names, threshold)


print('Recognition functions defined.')

Recognition functions defined.


## Cell 4: Load Known Faces (Subdirectory Structure) & YOLO

In [30]:
def load_known_faces(directory):
    """
    Load known faces from subdirectory structure:
        known_faces/
            PersonName/
                img1.jpg
                img2.jpg
                ...

    For each person, computes one embedding per image then averages them
    into a single representative embedding. This makes recognition more
    robust across different angles, lighting, expressions.

    Returns:
        known_face_encodings : list of averaged 512-d embeddings (one per person)
        known_face_names     : list of person name strings
    """
    encodings, names = [], []

    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created '{directory}'. Add person subdirectories and re-run.")
        return encodings, names

    for person_name in sorted(os.listdir(directory)):
        person_dir = os.path.join(directory, person_name)

        # Skip files at root level — only process subdirectories
        if not os.path.isdir(person_dir):
            continue

        person_embeddings = []

        for filename in os.listdir(person_dir):
            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img_path = os.path.join(person_dir, filename)
            try:
                objs = DeepFace.represent(
                    img_path=img_path,
                    model_name='ArcFace',
                    enforce_detection=False
                )
                person_embeddings.append(np.array(objs[0]['embedding']))
                print(f'  [{person_name}] Loaded: {filename}')
            except Exception as e:
                print(f'  [{person_name}] Skipped {filename}: {e}')

        if len(person_embeddings) == 0:
            print(f'  [{person_name}] No valid images found — skipping.')
            continue

        # Average all embeddings for this person into one representative vector
        avg_embedding = np.mean(person_embeddings, axis=0)

        # Normalise the averaged embedding
        avg_embedding = avg_embedding / (np.linalg.norm(avg_embedding) + 1e-9)

        encodings.append(avg_embedding)
        names.append(person_name)
        print(f'  [{person_name}] Averaged {len(person_embeddings)} image(s) → 1 embedding')

    print(f'\nTotal people loaded: {len(names)}')
    print(f'Names: {names}')
    return encodings, names


def load_yolo_model(model_path):
    print(f'Loading YOLO from {model_path} ...')
    model = YOLO(model_path)
    print('YOLO loaded.')
    return model


# ── Run loaders ────────────────────────────────────────────────────────────
print('=== Loading Known Faces ===')
known_face_encodings, known_face_names = load_known_faces(KNOWN_FACES_DIR)

print('\n=== Loading YOLO ===')
yolo_model = load_yolo_model(MODEL_PATH)

=== Loading Known Faces ===
  [Ayushi] Loaded: tyityiti.jpeg
  [Ayushi] Averaged 1 image(s) → 1 embedding
  [Divya] Loaded: WhatsApp Image 2026-03-23 at 4.21.21 PM.jpeg
  [Divya] Averaged 1 image(s) → 1 embedding
  [Harshita] Loaded: WhatsApp Image 2026-03-23 at 4.29.17 PM.jpeg
  [Harshita] Averaged 1 image(s) → 1 embedding
  [Lavanya] Loaded: WhatsApp Image 2026-03-23 at 4.29.16 PM.jpeg
  [Lavanya] Averaged 1 image(s) → 1 embedding
  [Mohit] Loaded: WIN_20260301_21_17_38_Pro.jpg
  [Mohit] Averaged 1 image(s) → 1 embedding
  [Muskan] Loaded: Muskan.jpg
  [Muskan] Averaged 1 image(s) → 1 embedding
  [Nayan] Loaded: Nayan.jpg
  [Nayan] Averaged 1 image(s) → 1 embedding
  [Piyush] Loaded: WhatsApp Image 2026-03-23 at 4.46.07 PM.jpeg
  [Piyush] Averaged 1 image(s) → 1 embedding
  [Ruchika] Loaded: WhatsApp Image 2026-03-23 at 4.29.11 PM.jpeg
  [Ruchika] Averaged 1 image(s) → 1 embedding
  [Shami] Loaded: WhatsApp Image 2026-03-23 at 4.29.11 PM.jpeg
  [Shami] Averaged 1 image(s) → 1 embeddi

## Cell 5: YOLO Detection & Tracker Management

In [ ]:
def run_yolo_detection(frame, model, conf_threshold):
    """Run YOLO, return list of (x1,y1,x2,y2) face boxes."""
    results = model.predict(source=frame, conf=conf_threshold,
                            save=False, device=0, verbose=False)
    boxes = []
    for box in results[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
        boxes.append(clamp_box((x1,y1,x2,y2), frame.shape))
    return boxes


def create_csrt_tracker():
    try:
        return cv2.legacy.TrackerCSRT_create()
    except AttributeError:
        return cv2.TrackerCSRT_create()


def build_multitracker(frame, boxes_xyxy):
    """Build a fresh MultiTracker from a list of xyxy boxes."""
    try:
        mt = cv2.legacy.MultiTracker_create()
    except AttributeError:
        mt = cv2.MultiTracker_create()
    for box in boxes_xyxy:
        mt.add(create_csrt_tracker(), frame, xyxy_to_xywh(box))
    return mt


def update_multitracker(mt, frame):
    """Update tracker, return list of xyxy boxes."""
    result = mt.update(frame)
    if isinstance(result, tuple) and len(result) == 2:
        _, raw = result
    else:
        raw = result
    return [xywh_to_xyxy(b) for b in raw]


def is_persisted(label):
    """Return True if label is an orange/persisted name (tilde prefix)."""
    return '~' in label


def deduplicate_face_pool(face_pool, iou_threshold=0.45):
    """
    After every frame, remove any duplicate boxes that overlap the same face.
    When two entries overlap above iou_threshold, the lower-priority one is dropped:
        Priority: green (3) > orange (2) > red (1)
    If same priority, keep the one recognised more recently.
    """
    def priority(entry):
        label = entry['label']
        if is_unknown(label):   return 1   # red
        if is_persisted(label): return 2   # orange
        return 3                           # green

    remove = set()
    for i in range(len(face_pool)):
        if i in remove:
            continue
        for j in range(i + 1, len(face_pool)):
            if j in remove:
                continue
            if compute_iou(face_pool[i]['box'], face_pool[j]['box']) >= iou_threshold:
                # Keep higher priority; if tied keep more recently recognised
                pi, pj = priority(face_pool[i]), priority(face_pool[j])
                if pi >= pj:
                    remove.add(j)
                else:
                    remove.add(i)
                    break   # i is gone, stop inner loop

    return remove


def _yolo_face_in_region(frame, box, model, conf_threshold):
    """
    Run YOLO on the padded region around 'box'.
    Returns (face_found: bool, snapped_box: tuple|None).
    snapped_box is the best YOLO detection in FULL-FRAME coordinates, or None.
    """
    x1, y1, x2, y2 = box
    if x2 <= x1 or y2 <= y1:
        return False, None

    h, w = frame.shape[:2]
    pad  = 20
    rx1  = max(0, x1 - pad)
    ry1  = max(0, y1 - pad)
    rx2  = min(w, x2 + pad)
    ry2  = min(h, y2 + pad)

    region = frame[ry1:ry2, rx1:rx2]
    if region.size == 0:
        return False, None

    results = model.predict(source=region, conf=conf_threshold,
                            save=False, verbose=False)
    if len(results[0].boxes) == 0:
        return False, None

    # Pick the detection with highest confidence
    best = max(results[0].boxes, key=lambda b: float(b.conf[0]))
    bx1, by1, bx2, by2 = best.xyxy[0].cpu().numpy().astype(int)
    # Convert back to full-frame coordinates
    snapped = clamp_box((rx1 + bx1, ry1 + by1, rx1 + bx2, ry1 + by2), frame.shape)
    return True, snapped


def per_frame_yolo_maintenance(frame, face_pool, model, conf_threshold):
    """
    Called every frame BEFORE re-recognition.
    Implements per-colour-state YOLO behaviour:

    GREEN  → snap box to YOLO detection if found (keep tight).
             Green boxes are NEVER removed here.

    ORANGE → YOLO every frame:
               • face found  → snap box to YOLO detection, keep entry
               • no face     → mark for removal (tracker drifted, person gone)

    RED    → YOLO every frame:
               • face found  → snap box to YOLO detection, keep entry
               • no face     → mark for removal (ghost box)

    Returns set of indices to remove from face_pool.
    """
    remove_indices = set()

    for i, entry in enumerate(face_pool):
        label = entry['label']
        is_green  = not (is_unknown(label) or is_persisted(label))
        is_orange = is_persisted(label)
        is_red    = is_unknown(label)

        # Lower confidence threshold for orange (partial face tolerated)
        check_conf = (conf_threshold * 0.75) if is_orange else conf_threshold

        face_found, snapped = _yolo_face_in_region(
            frame, entry['box'], model, check_conf
        )

        GREEN_YOLO_MISS_LIMIT = 5  # remove green box after this many consecutive YOLO misses

        if is_green:
            if face_found and snapped:
                entry['box'] = snapped
                entry['green_yolo_miss'] = 0   # reset miss counter on detection
            else:
                entry['green_yolo_miss'] = entry.get('green_yolo_miss', 0) + 1
                if entry['green_yolo_miss'] >= GREEN_YOLO_MISS_LIMIT:
                    remove_indices.add(i)       # too many misses → remove green box

        elif is_orange:
            if face_found and snapped:
                entry['box'] = snapped   # keep box tight
            else:
                remove_indices.add(i)    # no face → tracker ghost, remove

        elif is_red:
            if face_found and snapped:
                entry['box'] = snapped   # keep box tight
            else:
                remove_indices.add(i)    # no face → ghost, remove

    return remove_indices



print('Detection & tracking functions defined.')


Detection & tracking functions defined.


## Cell 6: Pipeline State & Per-Face Tracking Objects

In [32]:
"""
Per-face state is stored as a list of dicts — one entry per tracked face:

  face_pool = [
    {
      'box'            : (x1,y1,x2,y2)   current tracked position
      'label'          : str              display label  e.g. 'Mohit (0.18)'
      'confirmed_name' : str | None       last KNOWN name (persists over Unknown)
      'last_rerecog'   : float            timestamp face last entered the queue
      'in_queue'       : bool             True while face is waiting in rerecog queue
    },
    ...
  ]

The MultiTracker index aligns with face_pool index.
When we rebuild the MultiTracker the face_pool is rebuilt in the same order.
"""

def init_pipeline_state():
    return {
        'multi_tracker'   : None,
        'face_pool'       : [],    # list of per-face dicts (see above)
        'candidate_faces' : {},    # unconfirmed detections waiting to enter pool
        'rerecog_queue'   : [],    # ordered list of face_pool indices awaiting re-recog
                                  # red indices are inserted at front, green/orange at back
    }


def make_face_entry(box, label, now):
    """Create a new face_pool entry."""
    confirmed = None if is_unknown(label) else label.split(' (')[0]
    return {
        'box'            : box,
        'label'          : label,
        'confirmed_name' : confirmed,
        'last_rerecog'   : now,
        'miss_count'      : 0,
        'green_yolo_miss' : 0,     # consecutive frames green box had no YOLO face
        'in_queue'        : False,  # not yet queued
    }


def update_face_entry_after_rerecog(entry, new_name, new_dist):
    """
    Update a face entry after a re-recognition attempt.

    PERSISTENCE RULE:
      - Successful recognition (not Unknown)
            → update label, confirmed_name; reset miss counter
      - Unknown returned BUT confirmed_name exists (face partially hidden)
            → keep old name as orange (~) label, increment miss counter
            → after UNKNOWN_MISS_LIMIT consecutive misses the name is cleared
              (true departure / different person)
      - Unknown AND no confirmed_name
            → stays red Unknown
    """
    UNKNOWN_MISS_LIMIT = 50   # consecutive Unknown results before clearing name

    if new_name != 'Unknown':
        # Successful re-recognition — restore green, reset misses
        entry['label']          = f"{new_name} ({new_dist:.2f})"
        entry['confirmed_name'] = new_name
        entry['miss_count']     = 0
    elif entry['confirmed_name'] is not None:
        # Face partially hidden / recognition uncertain — stay orange
        entry['miss_count'] = entry.get('miss_count', 0) + 1
        if entry['miss_count'] >= UNKNOWN_MISS_LIMIT:
            # Too many consecutive misses → person likely gone, clear name
            entry['label']          = f"Unknown ({new_dist:.2f})"
            entry['confirmed_name'] = None
            entry['miss_count']     = 0
        else:
            # Keep persisted name with miss counter shown
            remaining = UNKNOWN_MISS_LIMIT - entry['miss_count']
            entry['label'] = f"{entry['confirmed_name']} (~{entry['miss_count']}/{UNKNOWN_MISS_LIMIT})"
    else:
        # Genuinely unknown, no history
        entry['label']      = f"Unknown ({new_dist:.2f})"
        entry['miss_count'] = 0

    entry['last_rerecog'] = time.time()


def box_key(box, grid=20):
    """Quantise box to grid for stable candidate dict key."""
    return tuple(int(c // grid) for c in box)


def update_candidate_faces(state, det_boxes, confirm_frames):
    """
    Track consecutive-frame count for unconfirmed detections.
    Returns list of confirmed new face boxes ready to enter the pool.
    """
    candidates = state['candidate_faces']
    tracked    = [f['box'] for f in state['face_pool']]

    for k in candidates:
        candidates[k]['seen'] = False

    for det in det_boxes:
        if any(compute_iou(det, tb) >= IOU_THRESHOLD for tb in tracked):
            continue  # already tracked

        matched_key = None
        for ck, cv in candidates.items():
            if compute_iou(det, cv['box']) >= IOU_THRESHOLD:
                matched_key = ck
                break

        if matched_key:
            candidates[matched_key]['box']  = det
            candidates[matched_key]['hits'] += 1
            candidates[matched_key]['seen'] = True
        else:
            k = box_key(det)
            candidates[k] = {'box': det, 'hits': 1, 'seen': True}

    stale = [k for k,v in candidates.items() if not v.get('seen', False)]
    for k in stale:
        del candidates[k]

    confirmed, confirmed_keys = [], []
    for k, v in candidates.items():
        if v['hits'] >= confirm_frames:
            confirmed.append(v['box'])
            confirmed_keys.append(k)
    for k in confirmed_keys:
        del candidates[k]

    return confirmed


print('Pipeline state functions defined.')


Pipeline state functions defined.


## Cell 7: Per-Face Re-Recognition Logic

In [33]:
def build_rerecog_queue(face_pool, rerecog_queue, cooldown_sec):
    """
    Maintain the re-recognition queue each frame.

    Rules:
      - Every face (green, orange, red) should eventually be re-recognised.
      - A face enters the queue only if it is not already in it AND its
        cooldown since last re-recognition has passed.
      - RED faces are inserted at the FRONT (high priority).
      - GREEN and ORANGE faces are appended at the BACK (normal priority).
      - Orange faces ARE included — if re-recog succeeds they go green,
        if it fails the miss counter increments (they stay orange or clear).
      - One face is processed per frame to spread CPU load.

    Queue entries are face_pool indices (ints).
    Stale indices (face removed from pool) are silently skipped at pop time.
    """
    now = time.time()
    pool_ids = set(range(len(face_pool)))

    # Purge any stale indices no longer in pool
    rerecog_queue[:] = [idx for idx in rerecog_queue if idx in pool_ids]

    queued_set = set(rerecog_queue)

    for idx, entry in enumerate(face_pool):
        if idx in queued_set:
            continue                             # already waiting
        if entry.get('in_queue', False):
            continue
        if (now - entry['last_rerecog']) < cooldown_sec:
            continue                             # still in cooldown

        label    = entry['label']
        is_red   = is_unknown(label)

        entry['in_queue'] = True
        if is_red:
            rerecog_queue.insert(0, idx)         # RED → front of queue
        else:
            rerecog_queue.append(idx)            # GREEN / ORANGE → back


def process_rerecog_queue(frame, face_pool, rerecog_queue,
                           known_encodings, known_names, threshold):
    """
    Pop ONE face from the front of the queue and run ArcFace on it.
    This keeps CPU load flat — exactly one recognition call per frame.

    State transitions after recognition:
      GREEN  → success : stay green (name + dist updated)
             → fail    : go orange  (name persisted, miss counter +1)
      ORANGE → success : go green   (confirmed again)
             → fail    : miss counter +1; clear to red after UNKNOWN_MISS_LIMIT
      RED    → success : go green   (identity found)
             → fail    : stay red   (still unknown)
    """
    while rerecog_queue:
        idx = rerecog_queue.pop(0)
        if idx >= len(face_pool):
            continue                             # stale index, skip

        entry = face_pool[idx]
        entry['in_queue'] = False

        x1, y1, x2, y2 = entry['box']
        face_crop = frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            entry['last_rerecog'] = time.time()
            return                               # used our one slot, done

        new_name, new_dist = recognize_face(
            face_crop, known_encodings, known_names, threshold
        )
        update_face_entry_after_rerecog(entry, new_name, new_dist)
        return                                   # one face per frame — stop here


def rebuild_pool_from_yolo(frame, yolo_boxes, old_face_pool,
                           known_encodings, known_names,
                           threshold, now):
    """
    Called only when we need a full tracker rebuild (e.g. tracker crashed).
    Matches YOLO boxes to existing pool entries by IOU to reuse confirmed_name,
    then re-recognises only faces that have no confirmed_name yet.
    Returns (multi_tracker, new_face_pool).
    """
    if len(yolo_boxes) == 0:
        return None, []

    new_pool = []
    used_old = set()

    for box in yolo_boxes:
        # Try to match to an existing pool entry by IOU
        best_idx, best_iou = -1, IOU_THRESHOLD
        for i, old in enumerate(old_face_pool):
            if i in used_old:
                continue
            iou = compute_iou(box, old['box'])
            if iou > best_iou:
                best_iou = iou
                best_idx = i

        if best_idx >= 0:
            # Reuse existing entry — update box, keep name history
            entry = dict(old_face_pool[best_idx])
            entry['box'] = box
            used_old.add(best_idx)
        else:
            # Brand new face — recognise immediately
            x1,y1,x2,y2 = box
            crop = frame[y1:y2, x1:x2]
            name, dist = ('Unknown', 1.0) if crop.size == 0 else \
                         recognize_face(crop, known_encodings, known_names, threshold)
            entry = make_face_entry(box, f"{name} ({dist:.2f})", now)

        new_pool.append(entry)

    mt = build_multitracker(frame, yolo_boxes)
    return mt, new_pool


print('Per-face re-recognition functions defined.')


Per-face re-recognition functions defined.


## Cell 8: Drawing / Visualisation

In [34]:
def draw_tracked_faces(frame, face_pool, rerecog_queue):
    """
    Draw each tracked face with:
      - Green box  = confirmed known person
      - Orange box = persisted name (re-recog returned Unknown but old name kept)
      - Red box    = genuinely Unknown
      - Small countdown showing seconds until next re-recognition for that face
    """
    now = time.time()
    for entry in face_pool:
        x1,y1,x2,y2 = entry['box']
        label = entry['label']

        if is_unknown(label):
            color = (0, 0, 255)       # red
        elif '~' in label:
            color = (0, 140, 255)     # orange — persisted name
        else:
            color = (0, 220, 0)       # green — confirmed

        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

        # Name label
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1-22), (x1+tw+4, y1), color, -1)
        cv2.putText(frame, label, (x1+2, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)

        # Show queue position so you can see re-recog scheduling live
        pool_idx = face_pool.index(entry)
        if pool_idx in rerecog_queue:
            q_pos   = rerecog_queue.index(pool_idx)
            cd_text = f"q:{q_pos+1}/{len(rerecog_queue)}"
        elif '~' in label:
            cd_text = "yolo"
        elif not is_unknown(label) and not is_persisted(label):
            # Green box: show YOLO miss counter if face is disappearing
            miss = entry.get('green_yolo_miss', 0)
            if miss > 0:
                cd_text = f"no-face:{miss}/10"
            else:
                cd_text = "queued" if entry.get('in_queue') else "wait"
        else:
            cd_text = "queued" if entry.get('in_queue') else "wait"
        cv2.putText(frame, cd_text, (x1+2, y2-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1)


def draw_candidate_faces(frame, candidate_faces, confirm_frames):
    """Yellow boxes for unconfirmed detections with hit counter."""
    for v in candidate_faces.values():
        x1,y1,x2,y2 = v['box']
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,215,255), 1)
        cv2.putText(frame, f"Checking {v['hits']}/{confirm_frames}",
                    (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0,215,255), 1)


def draw_overlay_info(frame, fps, num_tracked):
    cv2.putText(frame, f"FPS: {fps:.1f}",       (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (50,200,50), 2)
    cv2.putText(frame, f"Tracked: {num_tracked}",(10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (50,200,50), 2)


print('Drawing functions defined.')


Drawing functions defined.


## Cell 9: Main Processing Loop

**Per-frame flow:**
1. Run YOLO on every frame
2. Update MultiTracker → get new box positions
3. Sync face_pool boxes from tracker output + **snap to nearest YOLO box** for accuracy
4. Run per-face re-recognition for any face whose timer has expired (independent per face)
5. **Validate all boxes with YOLO** — remove Unknown/orange if no face found immediately; remove **green** boxes after **10 consecutive frames** with no YOLO detection
6. Check YOLO for new untracked faces → confirm over N frames → add to pool
7. Draw & display


In [35]:
VIDEO_SOURCE = r'C:\Users\DELL\Downloads\Phone Link\video.mp4'
#VIDEO_SOURCE = 0
cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    raise RuntimeError(f'Cannot open video source: {VIDEO_SOURCE}')

state     = init_pipeline_state()
prev_time = time.time()

print("Pipeline running. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print('End of stream.')
        break

    now = time.time()
    fps = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now

    # ── Step 1: YOLO detection every frame ───────────────────────────────
    det_boxes = run_yolo_detection(frame, yolo_model, CONF_THRESHOLD)

    # ── Step 2: Update MultiTracker ──────────────────────────────────────
    if state['multi_tracker'] is not None and len(state['face_pool']) > 0:
        updated_boxes = update_multitracker(state['multi_tracker'], frame)

        if len(updated_boxes) == len(state['face_pool']):
            # Step 3: Sync tracker positions back into face_pool
            for entry, new_box in zip(state['face_pool'], updated_boxes):
                entry['box'] = new_box
            # Step 3b: YOLO per-frame maintenance handles snapping below
        else:
            # Tracker count mismatch — rebuild from YOLO
            mt, new_pool = rebuild_pool_from_yolo(
                frame, det_boxes, state['face_pool'],
                known_face_encodings, known_face_names,
                RECOGNITION_THRESHOLD, now
            )
            state['multi_tracker'] = mt
            state['face_pool']     = new_pool

    # ── Step 4: Queue-based re-recognition (1 face per frame) ───────────
    #   All faces cycle through the queue; red faces jump to the front.
    #   Transitions: green→green/orange | orange→green/deleted | red→green/deleted
    build_rerecog_queue(
        state['face_pool'], state['rerecog_queue'], RERECOG_QUEUE_COOLDOWN
    )
    process_rerecog_queue(
        frame, state['face_pool'], state['rerecog_queue'],
        known_face_encodings, known_face_names, RECOGNITION_THRESHOLD
    )

    # ── Step 4b: YOLO per-frame maintenance ──────────────────────────────
    #   • GREEN  → snap box to YOLO detection (never removed)
    #   • ORANGE → snap box + remove if no face found
    #   • RED    → snap box + remove if no face found
    remove_indices = per_frame_yolo_maintenance(
        frame, state['face_pool'], yolo_model, CONF_THRESHOLD
    )
    if remove_indices:
        state['face_pool'] = [
            e for i, e in enumerate(state['face_pool'])
            if i not in remove_indices
        ]
        surviving_boxes = [f['box'] for f in state['face_pool']]
        if surviving_boxes:
            state['multi_tracker'] = build_multitracker(frame, surviving_boxes)
        else:
            state['multi_tracker'] = None

    # ── Step 4c: Remove duplicate boxes on the same face ─────────────────
    dup_indices = deduplicate_face_pool(state['face_pool'])
    if dup_indices:
        state['face_pool'] = [
            e for i, e in enumerate(state['face_pool'])
            if i not in dup_indices
        ]
        surviving_boxes = [f['box'] for f in state['face_pool']]
        if surviving_boxes:
            state['multi_tracker'] = build_multitracker(frame, surviving_boxes)
        else:
            state['multi_tracker'] = None


    # ── Step 5a: Find new untracked faces from YOLO ───────────────────────
    confirmed_new = update_candidate_faces(
        state, det_boxes, NEW_FACE_CONFIRM_FRAMES
    )

    # ── Step 5b: Add confirmed new faces to pool + tracker ────────────────
    for new_box in confirmed_new:
        x1,y1,x2,y2 = new_box
        crop = frame[y1:y2, x1:x2]
        name, dist = recognize_face(
            crop, known_face_encodings, known_face_names, RECOGNITION_THRESHOLD
        ) if crop.size > 0 else ('Unknown', 1.0)

        new_entry = make_face_entry(new_box, f"{name} ({dist:.2f})", now)
        state['face_pool'].append(new_entry)

        # Rebuild MultiTracker to include the new face
        all_boxes = [f['box'] for f in state['face_pool']]
        state['multi_tracker'] = build_multitracker(frame, all_boxes)

    # ── Handle first frame (pool still empty, faces detected) ────────────
    if state['multi_tracker'] is None and len(det_boxes) > 0:
        mt, new_pool = rebuild_pool_from_yolo(
            frame, det_boxes, [],
            known_face_encodings, known_face_names,
            RECOGNITION_THRESHOLD, now
        )
        state['multi_tracker'] = mt
        state['face_pool']     = new_pool

    # ── Step 6: Draw ──────────────────────────────────────────────────────
    draw_tracked_faces(frame, state['face_pool'], state['rerecog_queue'])
    draw_candidate_faces(frame, state['candidate_faces'], NEW_FACE_CONFIRM_FRAMES)
    draw_overlay_info(frame, fps, len(state['face_pool']))

    cv2.imshow('Face Detection + Tracking + Recognition v2', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Done.')


Pipeline running. Press 'q' to quit.
Done.
